# Classification metrics from hard labels

_Metrics & Evaluation — measuring models, data & statistics_

**Once a model commits to yes-or-no, four counts tell you everything.**

Every yes/no prediction lands in one of four boxes: right-positive, right-negative, wrong-positive, wrong-negative.

       Count how many predictions fall in each box. That table of four counts is the confusion matrix.

---

This notebook is a practice scaffold for the **Classification metrics from hard labels** lesson. We build the reference code up one step at a time — run each cell, read the short note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — scikit-learn

### Step 1 — Train a classifier and get hard labels

We load the breast-cancer data, split it into train and test sets (stratified so both classes stay balanced), and fit a logistic regression. The key line is `clf.predict`, which returns **hard 0/1 labels** — not probabilities. Hard-label metrics are exactly the ones that work on these committed yes/no decisions.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.33, random_state=0, stratify=y)

clf = LogisticRegression(max_iter=10000).fit(X_tr, y_tr)
pred = clf.predict(X_te)            # hard labels (0/1), not probabilities

### Step 2 — Build the confusion matrix and per-class metrics

The confusion matrix is the table of four counts everything else is derived from. In sklearn's breast-cancer data `0` means *malignant*, so we treat malignant as the **positive class** — then "recall" reads as "fraction of tumors we caught". Precision, recall, F1, F-beta, balanced accuracy, MCC, and Cohen's kappa are each just different ways of summarizing those four counts.

In [ ]:
from sklearn.metrics import (
    precision_score, recall_score, f1_score, fbeta_score,
    balanced_accuracy_score, matthews_corrcoef, cohen_kappa_score,
    confusion_matrix,
)

# In sklearn's breast-cancer data 0 = malignant. Treat malignant as the
# positive class so "recall" means "fraction of tumors we caught".
pos = 0

print("confusion matrix [rows=true, cols=pred]:")
print(confusion_matrix(y_te, pred, labels=[0, 1]))

print("precision (PPV) :", precision_score(y_te, pred, pos_label=pos))
print("recall (TPR)    :", recall_score(y_te, pred, pos_label=pos))
print("F1              :", f1_score(y_te, pred, pos_label=pos))
print("F-beta (beta=2) :", fbeta_score(y_te, pred, beta=2, pos_label=pos))
print("balanced acc    :", balanced_accuracy_score(y_te, pred))
print("MCC             :", matthews_corrcoef(y_te, pred))
print("Cohen's kappa   :", cohen_kappa_score(y_te, pred))

### Step 3 — Averaging variants and the full report

When you have more than one class (or want one summary number), you must decide *how to average* the per-class scores. **Macro** treats every class equally, **micro** pools all the counts first, and **weighted** weights each class by its size. `classification_report` prints the per-class table so you can see where the averages come from.

In [ ]:
from sklearn.metrics import classification_report

# averaging variants for the (here binary, but works for multi-class) report
print(classification_report(y_te, pred, digits=3))          # per-class table
print("macro F1   :", f1_score(y_te, pred, average="macro"))
print("micro F1   :", f1_score(y_te, pred, average="micro"))
print("weighted F1:", f1_score(y_te, pred, average="weighted"))

## Visualize the data & results

_How do the hard-label metrics compare for a real logistic-regression tumor classifier?_

### Step 1 — Fit a deliberately weaker model

To make the metrics *spread out* and differ from one another, we fit on only three raw features instead of all thirty. That yields a realistically imperfect classifier, so precision, recall, and the rest no longer all sit near the same value.

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X, y = load_breast_cancer(return_X_y=True)
# a few raw features -> a realistically imperfect model, so the metrics spread out
X = X[:, [0, 3, 7]]   # mean radius, mean area, mean concave points
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.33, random_state=0, stratify=y)

clf = LogisticRegression(max_iter=10000).fit(X_tr, y_tr)
pred = clf.predict(X_te)

### Step 2 — Unpack the four confusion-matrix counts

Calling `.ravel()` on the 2×2 confusion matrix flattens it into the four raw counts. We pass `labels=[1, 0]` so the order comes out as TN, FP, FN, TP relative to the positive (malignant) class. Every metric below is a different ratio of these same four numbers.

In [ ]:
from sklearn.metrics import confusion_matrix

pos = 0  # malignant is the positive class
tn, fp, fn, tp = confusion_matrix(y_te, pred, labels=[1, 0]).ravel()
print("TP,FP,FN,TN:", tp, fp, fn, tn)   # -> 55 8 15 110

### Step 3 — Compare the hard-label metrics side by side

Now read off each metric for this same set of predictions. Notice how recall (0.786) sits below precision (0.873) — the model misses some tumors — and how balanced accuracy and MCC give a more honest single number than plain accuracy when you care about the rarer class.

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    balanced_accuracy_score, matthews_corrcoef,
)

print(round(accuracy_score(y_te, pred), 3))                        # 0.878
print(round(precision_score(y_te, pred, pos_label=pos), 3))        # 0.873
print(round(recall_score(y_te, pred, pos_label=pos), 3))           # 0.786
print(round(f1_score(y_te, pred, pos_label=pos), 3))               # 0.827
print(round(balanced_accuracy_score(y_te, pred), 3))               # 0.859
print(round(matthews_corrcoef(y_te, pred), 3))                     # 0.735

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A classifier gives TP $= 40$, FP $= 10$, FN $= 20$, TN $= 130$. Find precision, recall, specificity, and accuracy.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Precision $= TP/(TP+FP) = 40/50$. — _Precision asks: of the positive flags, how many were right?_
- Recall $= TP/(TP+FN) = 40/60$. — _Recall asks: of the real positives, how many were caught?_
- Specificity $= TN/(TN+FP) = 130/140$. — _Specificity is recall for the negative class._
- Accuracy $= (TP+TN)/\text{all} = 170/200$. — _Accuracy is all correct over all predictions._

**Answer:** Precision $= 0.80$, recall $\approx 0.667$, specificity $\approx 0.929$, accuracy $= 0.85$.

</details>

**Problem 2.** A test set has 990 negatives and 10 positives. A model predicts "negative" for everything. What is its accuracy, and what is its recall?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Count the boxes: TP $= 0$, FP $= 0$, FN $= 10$, TN $= 990$. — _Predicting all-negative means zero positive flags._
- Accuracy $= (0+990)/1000 = 0.99$. — _It gets every negative right, and negatives dominate._
- Recall $= 0/(0+10) = 0$. — _It caught none of the 10 real positives._

**Answer:** Accuracy $= 0.99$ but recall $= 0$. This is exactly why accuracy lies under imbalance — report MCC or balanced accuracy instead.

</details>

**Problem 3.** Why can two classifiers have the same F1 but very different specificity?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Write F1 $= 2TP/(2TP+FP+FN)$. — _Notice TN never appears in the formula._
- Change TN while keeping TP, FP, FN fixed. — _F1 stays identical, but specificity $= TN/(TN+FP)$ changes._

**Answer:** F1 ignores true negatives entirely, so two models with the same TP/FP/FN but different TN share an F1 yet differ in specificity. Pair F1 with MCC or specificity to see the difference.

</details>